In [27]:
# Already installed
!pip3 install langchain langchain-community pypdf
!pip3 install sentence-transformers


Defaulting to user installation because normal site-packages is not writeable


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip
Defaulting to user installation because normal site-packages is not writeable


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip


In [15]:
import os
from langchain_community.document_loaders import PyPDFLoader

## Loading PDF Documents

In [16]:
def load_all_pdfs(folder_path):
 
    all_documents = []
 
    all_files = os.listdir(folder_path)
 
    all_files.sort()
 
    for filename in all_files:
 
        if not filename.lower().endswith(".pdf"):
            continue

        file_path = os.path.join(folder_path, filename)
        print(f"Loading: {filename}...")
 
        try:
            loader = PyPDFLoader(file_path)
            pages = loader.load()
 
            all_documents.extend(pages)
 
            print(f"Loaded {len(pages)} pages from {filename}")
 
        except Exception as e:
            print(f"Error loading {filename}: {e}")
 
    return all_documents

In [17]:
documents = load_all_pdfs("documents")
print(f"Total pages loaded: {len(documents)}")

Loading: 2759-7504-70-6-0400.pdf...
Loaded 8 pages from 2759-7504-70-6-0400.pdf
Loading: Alzheimer's extended.pdf...
Loaded 39 pages from Alzheimer's extended.pdf
Loading: Alzheimer's.pdf...
Loaded 31 pages from Alzheimer's.pdf
Loading: diabetes.pdf...
Loaded 20 pages from diabetes.pdf
Loading: medscimonit-30-e945091.pdf...
Loaded 7 pages from medscimonit-30-e945091.pdf
Total pages loaded: 105


In [18]:
for i, doc in enumerate(documents[:3]):
    print(f"Source: {doc.metadata['source']}, Page: {doc.metadata['page']}")
    print(doc.page_content[:300])
    print("---")

Source: documents/2759-7504-70-6-0400.pdf, Page: 0
400
Nishida Y, Watada H: The up-to-date for diabetes
Introduction
Diabetes mellitus is a chronic disease that continues 
to increase worldwide and requires proper manage-
ment and treatment. This review includes a general 
description of diabetes, its history, the basics of 
treatment, the latest tr
---
Source: documents/2759-7504-70-6-0400.pdf, Page: 1
401
Juntendo Medical Journal 70(6), 2024
levels. It is the only hormone that lowers blood 
glucose levels and is essential for incorporating 
glucose into the cells, using it as energy, or storing 
the excess glucose as lipids. 
Types of diabetes
There are two main types of diabetes: type 1 
diabete
---
Source: documents/2759-7504-70-6-0400.pdf, Page: 2
402
Nishida Y, Watada H: The up-to-date for diabetes
less to prevent complications 3).
Symptoms in diabetes
High blood glucose provokes symptoms of diabetes, 
such as thirst, polydipsia, polyuria, and weight loss. 
However, these symptom

## Chunking Documents

In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(documents)
 
print(f"\nBefore splitting: {len(documents)} pages")
print(f"After splitting:  {len(chunks)} chunks")
print(f"Average chunks per page: {len(chunks) / len(documents):.1f}")


Before splitting: 105 pages
After splitting:  839 chunks
Average chunks per page: 8.0


In [20]:
print("SAMPLE CHUNKS")
 
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i + 1} ---")
    print(f"Source: {chunk.metadata['source']}")
    print(f"Page:   {chunk.metadata['page']}")
    print(f"Length: {len(chunk.page_content)} characters")
    print(f"Text preview:")
    print(f"  {chunk.page_content[:200]}...")
    print()

SAMPLE CHUNKS

--- Chunk 1 ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   0
Length: 981 characters
Text preview:
  400
Nishida Y, Watada H: The up-to-date for diabetes
Introduction
Diabetes mellitus is a chronic disease that continues 
to increase worldwide and requires proper manage-
ment and treatment. This revi...


--- Chunk 2 ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   0
Length: 985 characters
Text preview:
  he suffered from various complications related to 
diabetes, such as thirst, blindness, and chest pain.
Basics of diabetes mellitus
What is diabetes?
Diabetes mellitus is a condition in which blood 
g...


--- Chunk 3 ---
Source: documents/2759-7504-70-6-0400.pdf
Page:   0
Length: 886 characters
Text preview:
  management and treatment advancements. Effective diabetes management includes maintaining blood glucose levels within 
normal ranges and monitoring HbA1c, a marker reflecting average glucose levels ov...



## Generating Embeddings for Chunks

In [24]:
from sentence_transformers import SentenceTransformer
import numpy as np 

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded!")


Model loaded!


In [28]:
chunk_texts = [chunk.page_content for chunk in chunks]
 
print(f"Embedding {len(chunk_texts)} chunks...")
print("(This may take 30-60 seconds depending on your machine)")

all_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=64
)

print(f"\n Embedding complete!")
print(f"  Chunks embedded: {len(all_embeddings)}")
print(f"  Embedding shape: {all_embeddings.shape}")
print(f"  Total memory: {all_embeddings.nbytes / 1024 / 1024:.1f} MB")

Embedding 839 chunks...
(This may take 30-60 seconds depending on your machine)

 Embedding complete!
  Chunks embedded: 839
  Embedding shape: (839, 384)
  Total memory: 1.2 MB
